# Probability and Statistics for AI and Mechanistic Interpretability

This notebook is a detailed, Colab-friendly guide to the probability and statistics that matter most for machine learning and interpretability.

## Why this matters

AI systems learn from data under uncertainty. Probability and statistics help you reason about:

- randomness in data
- uncertainty in model outputs
- distributions of features
- estimation from samples
- noise and variance
- likelihood-based learning
- Bayesian thinking
- information and surprise
- evaluation and hypothesis testing

Interpretability research also relies heavily on statistical thinking when you ask:

- is a discovered feature real or spurious?
- does an intervention truly change behavior?
- is a pattern stable across prompts and datasets?
- how much variance is explained by a direction or component?

## What this notebook covers

1. Probability basics
2. Events, sample spaces, and axioms
3. Conditional probability and Bayes' rule
4. Random variables
5. Expectation and variance
6. Covariance and correlation
7. Common distributions
8. Joint, marginal, and conditional distributions
9. Law of large numbers and central limit theorem
10. Sampling and estimation
11. Maximum likelihood estimation
12. Statistical inference and confidence intervals
13. Hypothesis testing
14. Basic Bayesian intuition
15. Entropy, cross-entropy, and KL divergence
16. Connection to deep learning and interpretability
17. Exercises

In [ ]:
import numpy as np
import torch
from math import factorial, exp, sqrt, pi, log

np.set_printoptions(suppress=True, precision=4)
torch.set_printoptions(precision=4, sci_mode=False)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

## 1. Probability: what is it?

Probability provides a mathematical language for uncertainty.

If an event $A$ can occur under some experiment, then its probability is written as:

$$
P(A)
$$

and satisfies:

$$
0 \le P(A) \le 1
$$

Interpretation:
- $0$: impossible
- $1$: certain
- values in between: uncertain

### In AI

Probability appears in:

- model outputs
- softmax distributions
- data modeling
- likelihood training
- Bayesian inference
- uncertainty estimation

## 2. Sample space and events

The **sample space** $\Omega$ is the set of all possible outcomes.

Example: rolling a fair die

$$
\Omega = \{1,2,3,4,5,6\}
$$

An **event** is a subset of $\Omega$.

Example:
- $A =$ “roll an even number” $= \{2,4,6\}$

### Kolmogorov axioms

A probability measure satisfies:

1. $P(A) \ge 0$
2. $P(\Omega) = 1$
3. For disjoint events $A_1, A_2, \dots$,
   $$
   P\left(\bigcup_i A_i\right) = \sum_i P(A_i)
   $$

### Consequences

- Complement rule:
  $$
  P(A^c) = 1 - P(A)
  $$
- Addition rule:
  $$
  P(A \cup B) = P(A) + P(B) - P(A \cap B)
  $$

In [ ]:
# Die example
omega = np.array([1, 2, 3, 4, 5, 6])

A = np.array([2, 4, 6])  # even
B = np.array([4, 5, 6])  # >= 4

P_A = len(A) / len(omega)
P_B = len(B) / len(omega)
P_A_intersection_B = len(np.intersect1d(A, B)) / len(omega)
P_A_union_B = P_A + P_B - P_A_intersection_B

print("P(A) =", P_A)
print("P(B) =", P_B)
print("P(A ∩ B) =", P_A_intersection_B)
print("P(A ∪ B) =", P_A_union_B)

## 3. Conditional probability

Conditional probability answers:

> What is the probability of $A$ given that $B$ has happened?

Definition:

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
\quad \text{for } P(B) > 0
$$

### Example

If you know the die roll is even, what is the probability it is also greater than 4?

Let:
- $A = \{4,5,6\}$
- $B = \{2,4,6\}$

Then:

$$
P(A \mid B) = \frac{P(A \cap B)}{P(B)}
= \frac{2/6}{3/6} = \frac{2}{3}
$$

### Why this matters in ML

A lot of ML is conditional modeling:

$$
P(y \mid x)
$$

For example:
- class probability given image
- next token probability given context
- output given hidden state

In [ ]:
P_A_given_B = P_A_intersection_B / P_B
print("P(A | B) =", P_A_given_B)

## 4. Independence

Events $A$ and $B$ are independent if:

$$
P(A \cap B) = P(A)P(B)
$$

This means knowledge of one event does not change the probability of the other.

### Important caution

Independence is a strong condition. In real data, many variables are not independent.

## 5. Bayes' rule

Bayes' rule is one of the most important equations in probability:

$$
P(A \mid B) = \frac{P(B \mid A) P(A)}{P(B)}
$$

This can be interpreted as:

$$
\text{posterior} = \frac{\text{likelihood} \times \text{prior}}{\text{evidence}}
$$

### Why Bayes' rule matters

It shows how new evidence updates prior beliefs.

### ML example

You may start with a prior belief about a class, then update it after seeing data.

### Interpretability intuition

If some activation pattern appears, how much should that change your belief that a concept is present? That is a Bayes-like question.

In [ ]:
# Simple disease testing example
P_disease = 0.01
P_positive_given_disease = 0.95
P_positive_given_no_disease = 0.05

P_no_disease = 1 - P_disease
P_positive = P_positive_given_disease * P_disease + P_positive_given_no_disease * P_no_disease

P_disease_given_positive = (P_positive_given_disease * P_disease) / P_positive

print("P(positive) =", P_positive)
print("P(disease | positive) =", P_disease_given_positive)

## 6. Random variables

A random variable maps outcomes to numbers.

### Discrete random variable

Takes countable values, e.g. die roll.

### Continuous random variable

Takes values on a continuum, e.g. height, time, activation magnitude.

We usually write a random variable as $X$.

### Probability mass function (PMF)

For discrete $X$:

$$
P(X = x)
$$

### Probability density function (PDF)

For continuous $X$, probabilities are given by integrals of a density $f(x)$:

$$
P(a \le X \le b) = \int_a^b f(x)\,dx
$$

## 7. Expectation

The expectation is the average value of a random variable.

### Discrete case

$$
\mathbb{E}[X] = \sum_x x\,P(X=x)
$$

### Continuous case

$$
\mathbb{E}[X] = \int_{-\infty}^{\infty} x f(x)\,dx
$$

Expectation is often called the **mean**.

### Why expectation matters in ML

Loss functions are often expectations:

$$
\mathcal{L}(\theta) = \mathbb{E}_{(x,y)\sim \mathcal{D}}[\ell(f_\theta(x), y)]
$$

In [ ]:
# Expected value of a fair die
values = np.array([1, 2, 3, 4, 5, 6])
probs = np.ones(6) / 6

expected_die = np.sum(values * probs)
print("E[X] for fair die =", expected_die)

## 8. Variance and standard deviation

Variance measures spread around the mean:

$$
\mathrm{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2]
$$

Equivalent form:

$$
\mathrm{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2
$$

Standard deviation:

$$
\mathrm{Std}(X) = \sqrt{\mathrm{Var}(X)}
$$

### Why this matters

Two variables can have the same mean but very different spread.

### In AI

Variance matters for:
- initialization
- gradient noise
- uncertainty
- feature scaling
- normalization

In [ ]:
E_X = np.sum(values * probs)
E_X2 = np.sum((values ** 2) * probs)
Var_X = E_X2 - E_X**2
Std_X = np.sqrt(Var_X)

print("Mean =", E_X)
print("Variance =", Var_X)
print("Std dev =", Std_X)

## 9. Covariance and correlation

For random variables $X$ and $Y$:

$$
\mathrm{Cov}(X, Y) = \mathbb{E}[(X - \mathbb{E}[X])(Y - \mathbb{E}[Y])]
$$

Covariance tells you whether variables move together.

- positive covariance: increase together
- negative covariance: one increases while the other decreases
- near zero: weak linear relationship

### Correlation

$$
\mathrm{Corr}(X,Y) = \frac{\mathrm{Cov}(X,Y)}{\mathrm{Std}(X)\mathrm{Std}(Y)}
$$

Correlation rescales covariance to lie between $-1$ and $1$.

### Why this matters in AI

- covariance matrices
- PCA
- redundant features
- measuring feature relationships
- representation geometry

In [ ]:
x = np.array([1., 2., 3., 4., 5.])
y = np.array([2., 4., 6., 8., 10.])
z = np.array([10., 8., 6., 4., 2.])

cov_xy = np.cov(x, y, bias=True)[0, 1]
corr_xy = np.corrcoef(x, y)[0, 1]

cov_xz = np.cov(x, z, bias=True)[0, 1]
corr_xz = np.corrcoef(x, z)[0, 1]

print("Cov(x, y) =", cov_xy, " Corr(x, y) =", corr_xy)
print("Cov(x, z) =", cov_xz, " Corr(x, z) =", corr_xz)

## 10. Important distributions

You do not need every distribution right now. But you should know the most important ones.

### Bernoulli distribution

A Bernoulli variable takes values in $\{0,1\}$:

$$
P(X=1)=p,\quad P(X=0)=1-p
$$

Mean:
$$
\mathbb{E}[X] = p
$$

Variance:
$$
\mathrm{Var}(X) = p(1-p)
$$

Used for yes/no events.

### Binomial distribution

If $X$ counts the number of successes in $n$ independent Bernoulli trials:

$$
P(X=k)=\binom{n}{k} p^k (1-p)^{n-k}
$$

### Gaussian / Normal distribution

$$
X \sim \mathcal{N}(\mu, \sigma^2)
$$

with density

$$
f(x) = \frac{1}{\sqrt{2\pi \sigma^2}}
\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)
$$

This is probably the most important continuous distribution in ML.

### Why Gaussian matters

- noise modeling
- initialization
- latent variable models
- central limit theorem
- many natural measurements

In [ ]:
# Bernoulli samples
np.random.seed(0)
bernoulli_samples = np.random.binomial(n=1, p=0.3, size=20)
print("Bernoulli samples:", bernoulli_samples)

# Binomial samples
binomial_samples = np.random.binomial(n=10, p=0.3, size=10)
print("Binomial samples:", binomial_samples)

# Gaussian samples
normal_samples = np.random.normal(loc=0.0, scale=1.0, size=10)
print("Gaussian samples:", normal_samples)

## 11. Joint, marginal, and conditional distributions

Suppose we have two variables $X$ and $Y$.

### Joint distribution

$$
P(X,Y)
$$

describes probabilities for pairs of values.

### Marginal distribution

To get the distribution of $X$ alone, sum over $Y$:

$$
P(X=x) = \sum_y P(X=x, Y=y)
$$

### Conditional distribution

$$
P(X=x \mid Y=y) = \frac{P(X=x, Y=y)}{P(Y=y)}
$$

### Why this matters

Most supervised learning is about conditionals:

$$
P(y \mid x)
$$

But generative modeling often tries to model a joint distribution:

$$
P(x, y)
$$

In [ ]:
# Small joint distribution table
joint = np.array([
    [0.1, 0.2],
    [0.3, 0.4]
])

# Rows correspond to X in {0,1}, columns to Y in {0,1}
P_X = joint.sum(axis=1)
P_Y = joint.sum(axis=0)

P_X_given_Y1 = joint[:, 1] / P_Y[1]

print("Joint distribution:\n", joint)
print("Marginal P(X):", P_X)
print("Marginal P(Y):", P_Y)
print("Conditional P(X | Y=1):", P_X_given_Y1)

## 12. Law of Large Numbers (LLN)

The Law of Large Numbers says that sample averages converge to the true expectation as the number of samples grows.

If $X_1, X_2, \dots, X_n$ are i.i.d. with mean $\mu$, then:

$$
\frac{1}{n}\sum_{i=1}^n X_i \to \mu
$$

as $n \to \infty$.

### Meaning

More data generally gives more stable estimates.

### Why this matters

Empirical averages in training approximate population expectations.

In [ ]:
np.random.seed(0)
samples = np.random.normal(loc=5.0, scale=2.0, size=10000)

running_means = np.cumsum(samples) / np.arange(1, len(samples) + 1)

print("First 5 running means:", running_means[:5])
print("Running mean after 100 samples:", running_means[99])
print("Running mean after 10000 samples:", running_means[-1])

## 13. Central Limit Theorem (CLT)

The CLT says that sums or averages of many independent random variables tend toward a Gaussian distribution, even when the original distribution is not Gaussian.

If $X_1, \dots, X_n$ are i.i.d. with mean $\mu$ and variance $\sigma^2$, then:

$$
\frac{\bar{X} - \mu}{\sigma / \sqrt{n}}
$$

approaches a standard normal distribution as $n$ grows.

### Why this matters

It explains why:
- Gaussian assumptions appear often
- confidence intervals work
- sampling distributions simplify

In [ ]:
np.random.seed(0)

# Start with an exponential distribution, which is not Gaussian
n_trials = 5000
sample_size = 30

sample_means = []
for _ in range(n_trials):
    sample = np.random.exponential(scale=1.0, size=sample_size)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print("Mean of sample means:", sample_means.mean())
print("Std of sample means:", sample_means.std())
print("Averages of non-Gaussian samples tend to look more Gaussian.")

## 14. Sampling and estimation

Usually we do not observe the true population distribution directly. We observe a sample.

From a sample $x_1, \dots, x_n$, we estimate quantities like:

### Sample mean

$$
\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i
$$

### Sample variance

A common estimator is:

$$
s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2
$$

### Why this matters

Training data is a sample from some underlying process. Your statistics are estimates, not perfect truths.

In [ ]:
sample = np.array([2.1, 1.9, 2.4, 2.0, 2.2])

sample_mean = np.mean(sample)
sample_var_unbiased = np.var(sample, ddof=1)
sample_var_population = np.var(sample, ddof=0)

print("Sample mean =", sample_mean)
print("Sample variance (ddof=1) =", sample_var_unbiased)
print("Population-style variance (ddof=0) =", sample_var_population)

## 15. Maximum Likelihood Estimation (MLE)

MLE chooses parameters that make the observed data most probable.

Suppose data points are $x_1, \dots, x_n$ and model parameters are $\theta$.

Likelihood:

$$
L(\theta) = P(x_1, \dots, x_n \mid \theta)
$$

Often we maximize the log-likelihood:

$$
\log L(\theta) = \sum_{i=1}^n \log P(x_i \mid \theta)
$$

### Why log-likelihood?

- products become sums
- numerically more stable
- easier to optimize

### Deep learning connection

Cross-entropy loss is closely related to negative log-likelihood.

In [ ]:
# MLE for mean of Gaussian with known variance:
# For Gaussian data, the MLE of mu is the sample mean.

np.random.seed(0)
data = np.random.normal(loc=3.0, scale=1.0, size=100)

mu_mle = np.mean(data)
print("MLE estimate for mu =", mu_mle)

## 16. Confidence intervals

A confidence interval gives a plausible range for a parameter based on the sample.

A common approximate 95% confidence interval for a mean is:

$$
\bar{x} \pm 1.96 \cdot \frac{s}{\sqrt{n}}
$$

where:
- $\bar{x}$ = sample mean
- $s$ = sample standard deviation
- $n$ = sample size

### Interpretation

Roughly speaking, under repeated sampling, 95% of such intervals would contain the true mean.

### Important caution

It does **not** mean “there is a 95% probability the true mean is in this specific interval” in the classical frequentist interpretation.

In [ ]:
sample = np.random.normal(loc=10.0, scale=2.0, size=50)

mean_est = sample.mean()
std_est = sample.std(ddof=1)
n = len(sample)

lower = mean_est - 1.96 * std_est / np.sqrt(n)
upper = mean_est + 1.96 * std_est / np.sqrt(n)

print("Mean estimate =", mean_est)
print("Approximate 95% confidence interval =", (lower, upper))

## 17. Hypothesis testing

Hypothesis testing asks whether observed data is compatible with a null hypothesis.

### Example structure

- Null hypothesis:
  $$
  H_0
  $$
- Alternative hypothesis:
  $$
  H_1
  $$

You compute a test statistic and assess whether the evidence is strong enough to reject $H_0$.

### Example intuition

Suppose you want to test whether a model intervention changes accuracy.

- $H_0$: no change
- $H_1$: change exists

### Important caution

A small p-value does not measure the size or importance of an effect. It measures incompatibility with the null model.

## 18. Basic Bayesian thinking

In Bayesian statistics, parameters are treated as uncertain and described by distributions.

We start with a prior:

$$
P(\theta)
$$

observe data $D$, and update to a posterior:

$$
P(\theta \mid D)
=
\frac{P(D \mid \theta)P(\theta)}{P(D)}
$$

### Why this matters conceptually

Bayesian thinking is useful whenever you want to combine:
- prior beliefs
- observed evidence
- uncertainty in estimates

## 19. Entropy

Entropy measures uncertainty in a distribution.

For a discrete random variable $X$:

$$
H(X) = -\sum_x P(x)\log P(x)
$$

### Intuition

- low entropy: predictable distribution
- high entropy: uncertain distribution

### Example

A fair coin has higher entropy than a coin that lands heads 99% of the time.

In [ ]:
def entropy(probs):
    probs = np.array(probs, dtype=float)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log(probs))

print("Entropy of fair coin:", entropy([0.5, 0.5]))
print("Entropy of biased coin:", entropy([0.99, 0.01]))

## 20. Cross-entropy

If the true distribution is $p$ and the model predicts $q$, cross-entropy is:

$$
H(p, q) = -\sum_x p(x)\log q(x)
$$

### In classification

If the true label is one-hot, minimizing cross-entropy encourages the model to assign high probability to the correct class.

This is one of the most common loss functions in deep learning.

In [ ]:
p = np.array([1.0, 0.0, 0.0])        # true one-hot label
q_good = np.array([0.9, 0.08, 0.02]) # good prediction
q_bad = np.array([0.2, 0.5, 0.3])    # bad prediction

def cross_entropy(p, q):
    eps = 1e-12
    q = np.clip(q, eps, 1.0)
    return -np.sum(p * np.log(q))

print("Cross-entropy (good q):", cross_entropy(p, q_good))
print("Cross-entropy (bad q):", cross_entropy(p, q_bad))

## 21. KL divergence

KL divergence measures how one distribution differs from another:

$$
D_{\mathrm{KL}}(p \| q)
=
\sum_x p(x)\log \frac{p(x)}{q(x)}
$$

### Properties

- $D_{\mathrm{KL}}(p \| q) \ge 0$
- equals $0$ only if $p=q$
- not symmetric:
  $$
  D_{\mathrm{KL}}(p \| q) \ne D_{\mathrm{KL}}(q \| p)
  $$

### Why this matters in ML

KL divergence appears in:
- variational autoencoders
- distribution matching
- policy optimization
- regularization

In [ ]:
def kl_divergence(p, q):
    eps = 1e-12
    p = np.clip(np.array(p, dtype=float), eps, 1.0)
    q = np.clip(np.array(q, dtype=float), eps, 1.0)
    return np.sum(p * np.log(p / q))

p = [0.7, 0.2, 0.1]
q = [0.5, 0.3, 0.2]

print("KL(p || q) =", kl_divergence(p, q))
print("KL(q || p) =", kl_divergence(q, p))

## 22. Probability in neural networks

### Softmax

For logits $z_1, \dots, z_K$:

$$
P(y=i \mid \mathbf{z}) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

This turns arbitrary real-valued scores into a probability distribution.

### Negative log-likelihood / cross-entropy

If the correct class is $y$, the loss is:

$$
\mathcal{L} = -\log P(y \mid \mathbf{z})
$$

This encourages the model to assign high probability to the correct outcome.

### In language models

The model predicts a distribution over next tokens:

$$
P(x_t \mid x_{<t})
$$

Training maximizes the likelihood of the training text.

In [ ]:
logits = torch.tensor([2.0, 0.5, -1.0])
probs = torch.softmax(logits, dim=0)

print("Logits:", logits)
print("Softmax probabilities:", probs)
print("Sum of probabilities:", probs.sum())

## 23. Statistical thinking in interpretability

Interpretability is not only conceptual and geometric. It is also statistical.

Questions you should ask:

1. **Stability**
   - Does a discovered direction work across datasets?
   - Or only on one prompt set?

2. **Variance**
   - How much does the effect size vary across runs or seeds?

3. **Signal vs noise**
   - Is the activation pattern meaningful, or random fluctuation?

4. **Generalization**
   - Does a probe or direction trained on one dataset work on new data?

5. **Intervention effect**
   - After steering an activation, is the observed change large and consistent?

These are fundamentally statistical questions.

## 24. Common pitfalls

### Pitfall 1: confusing probability with frequency only

Frequentist and Bayesian interpretations differ.

### Pitfall 2: treating confidence as certainty

A model assigning 0.99 probability can still be wrong.

### Pitfall 3: ignoring dependence

Many formulas assume independence; real-world data often violates it.

### Pitfall 4: interpreting p-values incorrectly

A p-value is not the probability that the null hypothesis is true.

### Pitfall 5: forgetting sample size

Small samples can produce unstable estimates.

## 25. Summary

The key probability and statistics ideas for AI are:

- probability measures uncertainty
- conditional probability underlies prediction
- Bayes' rule updates beliefs
- expectation is average behavior
- variance measures spread
- covariance and correlation capture dependence
- sampling creates estimation uncertainty
- MLE underlies many training objectives
- entropy measures uncertainty
- cross-entropy and KL divergence are core ML quantities

These ideas are essential for:
- classification
- language modeling
- uncertainty
- evaluation
- hypothesis testing
- representation analysis
- interpretability experiments

# Exercises

## Conceptual

1. Explain in your own words the difference between joint, marginal, and conditional probability.
2. Why does maximizing likelihood often become minimizing negative log-likelihood?
3. What does KL divergence measure, and why is it not symmetric?
4. Why does the central limit theorem matter in practical statistics?
5. Why is statistical thinking important in interpretability research?

## Computational

### Exercise 1: expectation and variance
Construct a discrete random variable with values:
$$
x = [0,1,2,3]
$$
and probabilities:
$$
p = [0.1, 0.2, 0.3, 0.4]
$$
Compute:
- expectation
- variance

### Exercise 2: Bayes' rule
Assume:
- $P(A)=0.2$
- $P(B|A)=0.8$
- $P(B|\neg A)=0.1$

Compute:
$$
P(A|B)
$$

### Exercise 3: sampling
Generate 1000 samples from a Gaussian with mean 5 and standard deviation 2. Estimate:
- sample mean
- sample variance
Compare them to the true values.

### Exercise 4: entropy
Compute the entropy of:
- a fair 4-way distribution
- a very peaked 4-way distribution

### Exercise 5: cross-entropy and KL
Choose two 3-class distributions and compute:
- entropy of $p$
- cross-entropy $H(p,q)$
- KL divergence $D_{\mathrm{KL}}(p \| q)$

### Exercise 6: softmax intuition
Take logits:
$$
[1.0, 2.0, 4.0]
$$
Compute softmax probabilities.
Then add 3 to all logits and recompute. What changed and why?

In [ ]:
# Starter workspace for exercises

import numpy as np
import torch

# Exercise 1
x = np.array([0., 1., 2., 3.])
p = np.array([0.1, 0.2, 0.3, 0.4])

# E = ...
# Var = ...

# Exercise 2
P_A = 0.2
P_B_given_A = 0.8
P_B_given_notA = 0.1

# Compute P(A|B)

# Exercise 3
np.random.seed(42)
samples = np.random.normal(loc=5.0, scale=2.0, size=1000)

# sample_mean = ...
# sample_var = ...

# Exercise 4
def entropy(probs):
    probs = np.array(probs, dtype=float)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log(probs))

# fair_entropy = ...
# peaked_entropy = ...

# Exercise 5
def cross_entropy(p, q):
    eps = 1e-12
    q = np.clip(np.array(q, dtype=float), eps, 1.0)
    p = np.array(p, dtype=float)
    return -np.sum(p * np.log(q))

def kl_divergence(p, q):
    eps = 1e-12
    p = np.clip(np.array(p, dtype=float), eps, 1.0)
    q = np.clip(np.array(q, dtype=float), eps, 1.0)
    return np.sum(p * np.log(p / q))

# p = ...
# q = ...

# Exercise 6
logits = torch.tensor([1.0, 2.0, 4.0])
# probs = torch.softmax(logits, dim=0)
# probs_shifted = torch.softmax(logits + 3.0, dim=0)

# Where to go next

After this notebook, the next natural math step is **calculus**, especially:

- derivatives
- gradients
- chain rule
- optimization
- Taylor approximation

Those are essential for understanding backpropagation and training dynamics.

For your interpretability goal, probability and statistics remain important because they help you judge:

- whether a feature is robust
- whether an intervention generalizes
- whether a result is noise or signal
- how much uncertainty remains